# NoseKnows — Gemma 4 E4B Fine-tuning

**Model:** `google/gemma-4-E4B-it` loaded with `AutoModelForCausalLM` (text-only, vision/audio encoders load but are never called)  
**Strategy:** QLoRA on the last 4 transformer layers + `lm_head` (r=16, alpha=32)  
**Data:** `dataset.jsonl` uploaded as a Kaggle dataset  
**Split:** 90% train / 10% validation  
**Epochs:** 3 with per-epoch checkpointing  
**Hardware:** Dual T4 GPU  

**Resume behaviour:** Re-run all cells. The trainer reads the latest checkpoint from `/kaggle/working/checkpoints/` automatically.  

**Outputs** (all in `/kaggle/working/`):  
- `checkpoints/` — per-epoch LoRA adapter checkpoints  
- `final_adapter/` — final LoRA adapter weights  
- `training_report.json` — training statistics  
- `plots/` — loss curves and metric visualisations

In [ ]:
# ── Cell 1: install dependencies ──────────────────────────────────────────
# trl>=0.8.6: SFTTrainer with apply_chat_template support
# peft>=0.10.0: LoraConfig with layer-specific targeting
# bitsandbytes==0.46.1: confirmed working with this transformers build
# matplotlib/seaborn: loss curve visualisations
import subprocess, sys

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "trl>=0.8.6",
        "peft>=0.10.0",
        "bitsandbytes==0.46.1",
        "accelerate>=0.29.0",
        "transformers>=4.45.0",
        "datasets>=2.18.0",
        "matplotlib",
        "seaborn",
    ],
    check=True,
)
print("Dependencies installed.")

In [ ]:
# ── Cell 2: imports ───────────────────────────────────────────────────────
import json
import logging
import os
import random
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import seaborn as sns
import torch
from datasets import Dataset
from kaggle_secrets import UserSecretsClient
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainerCallback,
    TrainerControl,
    TrainerState,
    TrainingArguments,
)
from trl import SFTTrainer, SFTConfig

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
)
log = logging.getLogger("nosknows_finetune")

# Seed everything for reproducibility.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — "
          f"{torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Cell 3: constants ─────────────────────────────────────────────────────

# ── paths ──
# UPDATE: set DATASET_PATH to the actual path after uploading dataset.jsonl
# as a Kaggle dataset. It will appear as:
# /kaggle/input/<your-dataset-name>/dataset.jsonl
DATASET_PATH    = Path("/kaggle/input/nosknows-dataset/dataset.jsonl")
OUTPUT_DIR      = Path("/kaggle/working")
CHECKPOINT_DIR  = OUTPUT_DIR / "checkpoints"
FINAL_ADAPTER   = OUTPUT_DIR / "final_adapter"
PLOTS_DIR       = OUTPUT_DIR / "plots"
REPORT_PATH     = OUTPUT_DIR / "training_report.json"

for d in [CHECKPOINT_DIR, FINAL_ADAPTER, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── model ──
MODEL_ID = "google/gemma-4-E4B-it"

# ── data ──
TRAIN_SPLIT     = 0.9       # 90% train, 10% validation
MAX_SEQ_LENGTH  = 512       # covers system + user + assistant for fragrance domain

# ── LoRA ──
# Targeting last 4 transformer layers + lm_head.
# Layer indices are resolved at runtime after the model is loaded.
LORA_R              = 16
LORA_ALPHA          = 32    # 2× r: standard scaling for stable training
LORA_DROPOUT        = 0.05  # light regularisation for small dataset
# All projection types within the targeted layers.
# gate_proj / up_proj / down_proj cover the MLP; q/k/v/o cover attention.
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
N_LORA_LAYERS       = 4     # how many layers from the end to target

# ── training ──
NUM_EPOCHS                  = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8     # effective batch = 2 × 8 × 2 GPUs = 32
LEARNING_RATE               = 2e-4
WARMUP_RATIO                = 0.05
LR_SCHEDULER                = "cosine"
FP16                        = True  # T4 supports fp16; bf16 is less stable on T4
# Save a checkpoint after every N optimizer steps (intermediate safety net).
# At ~253 steps/epoch, saving every 50 steps gives ~5 checkpoints per epoch.
SAVE_STEPS                  = 50
# Keep only the 3 most recent step-level checkpoints to save disk space.
# Per-epoch checkpoints are saved separately and always kept.
SAVE_TOTAL_LIMIT            = 3

print("Constants set.")
print(f"Dataset path : {DATASET_PATH}")
print(f"Model        : {MODEL_ID}")
print(f"LoRA r/alpha : {LORA_R}/{LORA_ALPHA}")
print(f"Max seq len  : {MAX_SEQ_LENGTH}")
print(f"Epochs       : {NUM_EPOCHS}")

In [ ]:
# ── Cell 4: load and prepare dataset ─────────────────────────────────────
#
# Each line of dataset.jsonl has:
#   {"messages": [{"role": ..., "content": ...}, ...], "_meta": {...}}
#
# We strip _meta and keep only the messages array.
# SFTTrainer with apply_chat_template reads the "messages" column directly.

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATASET_PATH}. "
        "Upload dataset.jsonl as a Kaggle dataset and update DATASET_PATH in Cell 3."
    )

log.info("Loading dataset from %s...", DATASET_PATH)
raw_records: list[dict] = []
with open(DATASET_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        # Strip _meta — not needed for training and not understood by SFTTrainer.
        raw_records.append({"messages": record["messages"]})

log.info("Loaded %d examples.", len(raw_records))

# Shuffle before splitting so the split is random but reproducible.
random.shuffle(raw_records)

split_idx    = int(len(raw_records) * TRAIN_SPLIT)
train_records = raw_records[:split_idx]
val_records   = raw_records[split_idx:]

train_dataset = Dataset.from_list(train_records)
val_dataset   = Dataset.from_list(val_records)

print(f"Train examples : {len(train_dataset):,}")
print(f"Val examples   : {len(val_dataset):,}")
print("\nSample record (first train example):")
for msg in train_dataset[0]["messages"]:
    print(f"  [{msg['role']}] {msg['content'][:100]}...")

In [ ]:
# ── Cell 5: load model and tokenizer ─────────────────────────────────────
#
# Loading google/gemma-4-E4B-it with AutoModelForCausalLM:
# The vision and audio encoders are part of the checkpoint and load into VRAM,
# but are never called during text-only fine-tuning. This costs ~1.5-2 GB extra
# VRAM vs a text-only checkpoint but uses the official Google weights.
#
# PYTORCH_ALLOC_CONF reduces fragmentation on T4 during weight loading.

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
if not hf_token:
    raise EnvironmentError(
        "HF_TOKEN is empty. Add it under Add-ons -> Secrets."
    )

log.info("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=hf_token,
    trust_remote_code=True,
)
# Gemma tokenizer may not have a pad token — set to eos if missing.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    log.info("Set pad_token = eos_token (%s)", tokenizer.eos_token)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # fp16 for T4 stability
    bnb_4bit_use_double_quant=True,
)

log.info("Loading model (4-bit NF4)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=hf_token,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    # No dtype argument — BitsAndBytesConfig controls dtype entirely.
)

# Disable model cache during training — not needed and wastes VRAM.
model.config.use_cache = False
# Required for gradient checkpointing compatibility with quantized models.
model.config.pretraining_tp = 1

log.info("Model loaded.")
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params / 1e9:.2f}B")
for i in range(torch.cuda.device_count()):
    alloc   = torch.cuda.memory_allocated(i) / 1e9
    reserved= torch.cuda.memory_reserved(i)  / 1e9
    print(f"GPU {i} ({torch.cuda.get_device_name(i)}): "
          f"allocated {alloc:.2f} GB | reserved {reserved:.2f} GB")

In [ ]:
# ── Cell 6: configure LoRA on last N transformer layers + lm_head ─────────
#
# We resolve layer names programmatically rather than hardcoding indices.
# This makes the script robust to different Gemma 4 architecture variants.
#
# Strategy:
#   1. Find the total number of transformer decoder layers.
#   2. Build the list of module names for the last N_LORA_LAYERS layers.
#   3. Add lm_head to the target list.
#   4. Pass the resolved names as layers_to_transform to LoraConfig.

def get_last_n_layer_names(model: AutoModelForCausalLM, n: int) -> list[str]:
    """
    Return the module name prefixes for the last n transformer decoder layers.

    Gemma 4 names its layers model.layers.N. We find the total layer count
    from model.config.num_hidden_layers, then build names for the last n.
    """
    try:
        num_layers = model.config.num_hidden_layers
    except AttributeError:
        # Some multimodal configs nest the text config.
        try:
            num_layers = model.config.text_config.num_hidden_layers
        except AttributeError:
            raise RuntimeError(
                "Cannot determine num_hidden_layers from model.config. "
                "Inspect model.config manually and set layer names explicitly."
            )
    start_layer = num_layers - n
    layer_names = [f"model.layers.{i}" for i in range(start_layer, num_layers)]
    log.info(
        "Total transformer layers: %d. Targeting last %d: %s ... %s",
        num_layers, n, layer_names[0], layer_names[-1],
    )
    return layer_names


target_layer_names = get_last_n_layer_names(model, N_LORA_LAYERS)

# layers_to_transform tells peft to only apply LoRA adapters within
# modules whose full name contains one of these prefixes.
# We also add lm_head explicitly via target_modules.
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    layers_to_transform=list(range(
        model.config.num_hidden_layers - N_LORA_LAYERS,
        model.config.num_hidden_layers,
    )),
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Sanity check: confirm trainable params are only in the last N layers.
print("\nTrainable modules:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  {name}")

In [ ]:
# ── Cell 7: training callbacks ────────────────────────────────────────────
#
# Two callbacks:
#   1. EpochCheckpointCallback: saves a named per-epoch checkpoint that is
#      never deleted by save_total_limit. This is the epoch-level safety net.
#   2. LossHistoryCallback: accumulates train and eval loss for visualisation.

class EpochCheckpointCallback(TrainerCallback):
    """
    Save a permanent per-epoch checkpoint to a named directory.
    These checkpoints are outside the standard checkpoint rotation managed
    by save_total_limit, so they are never automatically deleted.
    """

    def __init__(self, output_dir: Path) -> None:
        self.output_dir = output_dir

    def on_epoch_end(
        self,
        args: TrainingArguments,
        state: TrainerState,
        control: TrainerControl,
        **kwargs: Any,
    ) -> None:
        epoch = int(state.epoch)
        save_path = self.output_dir / f"epoch_{epoch}"
        save_path.mkdir(parents=True, exist_ok=True)
        # kwargs["model"] is the PEFT-wrapped model.
        kwargs["model"].save_pretrained(str(save_path))
        log.info("Epoch %d checkpoint saved to %s", epoch, save_path)


class LossHistoryCallback(TrainerCallback):
    """
    Collect training and validation loss at each logging step and epoch
    for post-training visualisation.
    """

    def __init__(self) -> None:
        self.train_losses: list[tuple[float, float]] = []  # (step, loss)
        self.eval_losses:  list[tuple[float, float]] = []  # (epoch, loss)

    def on_log(
        self,
        args: TrainingArguments,
        state: TrainerState,
        control: TrainerControl,
        logs: dict[str, float] | None = None,
        **kwargs: Any,
    ) -> None:
        if logs is None:
            return
        if "loss" in logs:
            self.train_losses.append((state.global_step, logs["loss"]))
        if "eval_loss" in logs:
            self.eval_losses.append((state.epoch, logs["eval_loss"]))


epoch_ckpt_cb  = EpochCheckpointCallback(CHECKPOINT_DIR)
loss_history_cb = LossHistoryCallback()

print("Callbacks defined.")

In [ ]:
# ── Cell 8: SFTConfig and SFTTrainer ──────────────────────────────────────
#
# SFTConfig wraps TrainingArguments with SFT-specific options.
# dataset_text_field is not used here because we use a messages column
# with apply_chat_template — SFTTrainer formats the conversation automatically.
#
# Checkpointing strategy:
#   - save_strategy="steps" with save_steps=SAVE_STEPS: intermediate step-level
#     checkpoints, rotated by save_total_limit=3 (disk safety)
#   - EpochCheckpointCallback: permanent per-epoch named checkpoint (never deleted)
#   - resume_from_checkpoint=True: if a checkpoint exists in output_dir,
#     the trainer resumes automatically on re-run

sft_config = SFTConfig(
    # ── output ──
    output_dir=str(CHECKPOINT_DIR),

    # ── training duration ──
    num_train_epochs=NUM_EPOCHS,

    # ── batch and accumulation ──
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    # ── optimisation ──
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER,
    optim="paged_adamw_8bit",  # memory-efficient optimiser for QLoRA

    # ── precision ──
    fp16=FP16,
    bf16=False,

    # ── sequence ──
    max_seq_length=MAX_SEQ_LENGTH,

    # ── checkpointing ──
    # Step-level: intermediate safety net, rotated
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    # Resume from latest step-level checkpoint on re-run
    resume_from_checkpoint=True,

    # ── evaluation ──
    eval_strategy="epoch",     # evaluate at end of each epoch
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # ── logging ──
    logging_dir=str(OUTPUT_DIR / "logs"),
    logging_strategy="steps",
    logging_steps=10,          # log train loss every 10 optimizer steps
    report_to="none",          # disable W&B / HF Hub reporting

    # ── SFT specific ──
    # Tell SFTTrainer to format using the model's chat template.
    # The dataset has a "messages" column with role/content dicts.
    dataset_kwargs={"skip_prepare_dataset": False},

    # ── misc ──
    seed=SEED,
    data_seed=SEED,
    remove_unused_columns=False,  # keep messages column for chat template
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    callbacks=[epoch_ckpt_cb, loss_history_cb],
)

print("Trainer configured.")
print(f"Steps per epoch (approx): {len(train_dataset) // (PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * torch.cuda.device_count())}")
print(f"Intermediate checkpoints every {SAVE_STEPS} steps (last {SAVE_TOTAL_LIMIT} kept).")
print(f"Permanent epoch checkpoints in: {CHECKPOINT_DIR}")

In [ ]:
# ── Cell 9: train ─────────────────────────────────────────────────────────
#
# If a step-level checkpoint exists in CHECKPOINT_DIR, the trainer resumes
# automatically from it (resume_from_checkpoint=True in SFTConfig).
# Per-epoch named checkpoints in CHECKPOINT_DIR/epoch_N/ are preserved
# regardless of save_total_limit.

log.info("Starting training...")
train_result = trainer.train()

log.info("Training complete.")
print(f"\nTraining summary:")
print(f"  Total steps          : {train_result.global_step}")
print(f"  Training loss (final): {train_result.training_loss:.4f}")
print(f"  Runtime              : {train_result.metrics.get('train_runtime', 0) / 3600:.2f}h")

In [ ]:
# ── Cell 10: save final adapter ───────────────────────────────────────────
#
# Saves only the LoRA adapter weights — not the full base model.
# The adapter is ~50-100 MB depending on r and number of targeted layers.
# Load it at inference time with:
#   from peft import PeftModel
#   model = PeftModel.from_pretrained(base_model, adapter_path)

log.info("Saving final LoRA adapter to %s...", FINAL_ADAPTER)
trainer.model.save_pretrained(str(FINAL_ADAPTER))
tokenizer.save_pretrained(str(FINAL_ADAPTER))
log.info("Final adapter saved.")

# Save training metrics to report.
report = {
    "model_id":            MODEL_ID,
    "lora_r":              LORA_R,
    "lora_alpha":          LORA_ALPHA,
    "lora_dropout":        LORA_DROPOUT,
    "n_lora_layers":       N_LORA_LAYERS,
    "num_epochs":          NUM_EPOCHS,
    "train_examples":      len(train_dataset),
    "val_examples":        len(val_dataset),
    "max_seq_length":      MAX_SEQ_LENGTH,
    "effective_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * torch.cuda.device_count(),
    "learning_rate":       LEARNING_RATE,
    "total_steps":         train_result.global_step,
    "final_train_loss":    train_result.training_loss,
    "runtime_hours":       train_result.metrics.get("train_runtime", 0) / 3600,
    "train_loss_history":  loss_history_cb.train_losses,
    "eval_loss_history":   loss_history_cb.eval_losses,
}
with open(REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)
log.info("Training report saved to %s", REPORT_PATH)

In [ ]:
# ── Cell 11: visualisations ───────────────────────────────────────────────
#
# Three plots:
#   1. Training loss curve (step-level)
#   2. Validation loss per epoch
#   3. Train vs validation loss comparison on the same axis

sns.set_theme(style="darkgrid", palette="muted")
FIGSIZE = (10, 5)

train_steps  = [s for s, _ in loss_history_cb.train_losses]
train_losses = [l for _, l in loss_history_cb.train_losses]
eval_epochs  = [e for e, _ in loss_history_cb.eval_losses]
eval_losses  = [l for _, l in loss_history_cb.eval_losses]

# ── Plot 1: training loss curve ──
fig, ax = plt.subplots(figsize=FIGSIZE)
ax.plot(train_steps, train_losses, linewidth=1.5, color="steelblue", label="Train loss")
ax.set_xlabel("Optimizer step", fontsize=12)
ax.set_ylabel("Loss", fontsize=12)
ax.set_title("NoseKnows — Training Loss Curve", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plot1_path = PLOTS_DIR / "train_loss_curve.png"
plt.savefig(plot1_path, dpi=150)
plt.show()
print(f"Saved: {plot1_path}")

# ── Plot 2: validation loss per epoch ──
if eval_losses:
    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.plot(
        eval_epochs, eval_losses,
        marker="o", linewidth=2, markersize=8,
        color="coral", label="Validation loss",
    )
    for x, y in zip(eval_epochs, eval_losses):
        ax.annotate(
            f"{y:.4f}",
            (x, y),
            textcoords="offset points",
            xytext=(0, 10),
            ha="center",
            fontsize=9,
        )
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Loss", fontsize=12)
    ax.set_title("NoseKnows — Validation Loss per Epoch", fontsize=14, fontweight="bold")
    ax.legend(fontsize=11)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    plt.tight_layout()
    plot2_path = PLOTS_DIR / "val_loss_per_epoch.png"
    plt.savefig(plot2_path, dpi=150)
    plt.show()
    print(f"Saved: {plot2_path}")

# ── Plot 3: train vs validation comparison ──
if eval_losses and train_losses:
    # Map eval epochs to the nearest training step for alignment.
    steps_per_epoch = max(train_steps) / NUM_EPOCHS if NUM_EPOCHS > 0 else 1
    eval_steps_approx = [e * steps_per_epoch for e in eval_epochs]

    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.plot(train_steps, train_losses, linewidth=1.2, color="steelblue",
            alpha=0.8, label="Train loss")
    ax.plot(eval_steps_approx, eval_losses, marker="o", linewidth=2,
            markersize=8, color="coral", label="Validation loss")
    ax.set_xlabel("Optimizer step", fontsize=12)
    ax.set_ylabel("Loss", fontsize=12)
    ax.set_title("NoseKnows — Train vs Validation Loss", fontsize=14, fontweight="bold")
    ax.legend(fontsize=11)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    plt.tight_layout()
    plot3_path = PLOTS_DIR / "train_vs_val_loss.png"
    plt.savefig(plot3_path, dpi=150)
    plt.show()
    print(f"Saved: {plot3_path}")

print("\nAll visualisations saved to", PLOTS_DIR)

In [ ]:
# ── Cell 12: inference sanity check ──────────────────────────────────────
#
# Run a quick inference check on three test queries to confirm the fine-tuned
# model produces NoseKnows-style responses before downloading the adapter.
#
# The model is already in eval mode after training (load_best_model_at_end=True
# loads the best checkpoint, which is in eval mode).

NOSKNOWS_SYSTEM_PROMPT = (
    "You are NoseKnows, a fragrance consultant who knows perfumery inside out. "
    "When someone describes what they are after, whether a mood, an occasion, "
    "or notes they love or cannot stand, you recommend real perfumes by name and "
    "brand and explain exactly why they fit, grounding your answer in the actual "
    "notes and accords. Warm, confident, specific. Never vague, never a catalogue. "
    "3 to 5 sentences."
)

TEST_QUERIES = [
    "I want something warm and cozy for autumn evenings, not too sweet.",
    "Looking for a fresh citrus scent for the office, nothing too loud.",
    "I love oud and rose but can't stand anything synthetic smelling.",
]


def run_inference(query: str) -> str:
    """Run a single text-only inference with the fine-tuned model."""
    messages = [
        {"role": "system",  "content": NOSKNOWS_SYSTEM_PROMPT},
        {"role": "user",    "content": query},
    ]
    # Use the tokenizer's chat template to format the input.
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    # Handle BatchEncoding vs plain tensor (same fix as generation script).
    if hasattr(input_ids, "input_ids"):
        input_ids = input_ids.input_ids
    elif isinstance(input_ids, dict):
        input_ids = input_ids["input_ids"]

    input_ids = input_ids.to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


print("=" * 60)
print("INFERENCE SANITY CHECK")
print("=" * 60)
for query in TEST_QUERIES:
    response = run_inference(query)
    print(f"\nUSER: {query}")
    print(f"NOSKNOWS: {response}")
    print("-" * 60)

In [ ]:
# ── Cell 13: final summary ────────────────────────────────────────────────

print("\n" + "=" * 62)
print("FINE-TUNING COMPLETE")
print("=" * 62)
print(f"  Model              : {MODEL_ID}")
print(f"  LoRA layers        : last {N_LORA_LAYERS} transformer layers + lm_head")
print(f"  LoRA r / alpha     : {LORA_R} / {LORA_ALPHA}")
print(f"  Train examples     : {len(train_dataset):,}")
print(f"  Val examples       : {len(val_dataset):,}")
print(f"  Epochs completed   : {NUM_EPOCHS}")
print(f"  Total steps        : {train_result.global_step}")
print(f"  Final train loss   : {train_result.training_loss:.4f}")
if loss_history_cb.eval_losses:
    best_val = min(l for _, l in loss_history_cb.eval_losses)
    print(f"  Best val loss      : {best_val:.4f}")
print(f"  Runtime            : {train_result.metrics.get('train_runtime', 0) / 3600:.2f}h")
print("=" * 62)
print(f"\nFinal adapter  -> {FINAL_ADAPTER}")
print(f"Epoch checkpoints -> {CHECKPOINT_DIR}/epoch_N/")
print(f"Plots          -> {PLOTS_DIR}")
print(f"Report         -> {REPORT_PATH}")
print("\nDownload final_adapter/ from the Output tab to use with the NoseKnows backend.")